In [2]:
import pandas as pd
import numpy as np

In [3]:
import os

PROJECT_ROOT = r"C:\Users\nicol\OneDrive\Bureau\Finance"
os.chdir(PROJECT_ROOT)

print("Current working directory:", os.getcwd())

Current working directory: C:\Users\nicol\OneDrive\Bureau\Finance


In [4]:
df = pd.read_csv("data/interim/market_clean.csv", index_col=0, parse_dates=True)
df.head(10)

,Open,High,Low,Close,Volume,log_return
Date,,,,,,
2015-01-05 00:00:00-05:00,2054.439941,2054.439941,2017.339966,2020.579956,3799120000,-0.018447
2015-01-06 00:00:00-05:00,2022.150024,2030.250000,1992.439941,2002.609985,4460110000,-0.008933
2015-01-07 00:00:00-05:00,2005.550049,2029.609985,2005.550049,2025.900024,3805480000,0.011563
2015-01-08 00:00:00-05:00,2030.609985,2064.080078,2030.609985,2062.139893,3934010000,0.017730
2015-01-09 00:00:00-05:00,2063.449951,2064.429932,2038.329956,2044.810059,3364140000,-0.008439
2015-01-12 00:00:00-05:00,2046.130005,2049.300049,2022.579956,2028.260010,3456460000,-0.008127
2015-01-13 00:00:00-05:00,2031.579956,2056.929932,2008.250000,2023.030029,4107300000,-0.002582
2015-01-14 00:00:00-05:00,2018.400024,2018.400024,1988.439941,2011.270020,4378680000,-0.005830
2015-01-15 00:00:00-05:00,2013.750000,2021.349976,1991.469971,1992.670044,4276720000,-0.009291


In [5]:
# Volatilité glissante
df["vol_5"]  = df["log_return"].rolling(5).std()
df["vol_10"] = df["log_return"].rolling(10).std()
df["vol_21"] = df["log_return"].rolling(21).std()

In [6]:
# Volatilité exponentielle
df["vol_ewma"] = df["log_return"].ewm(span=21).std()

In [7]:
from src.features.indicators import ATR

# Average True Range (ATR)
df["ATR"] = ATR(df)

In [8]:
import importlib
from src.features import indicators
importlib.reload(indicators)
from src.features.indicators import RSI

# Relative Strength Index (RSI)
df["RSI"] = RSI(df["Close"])

In [9]:
# MACD / Signal MACD
exp1 = df["Close"].ewm(span=12).mean()
exp2 = df["Close"].ewm(span=26).mean()
df["MACD"] = exp1 - exp2
df["MACD_signal"] = df["MACD"].ewm(span=9).mean()

In [10]:
# Volume logarithmique
df["log_volume"] = np.log(df["Volume"])

c:\Users\nicol\OneDrive\Bureau\Finance\.venv\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [11]:
# Variation du volume
df["volume_change"] = df["log_volume"].diff()

In [12]:
# Volalité réalisée cible
df["target_vol"] = df["log_return"].rolling(21).std().shift(-1)

In [13]:
df_features = df.dropna().copy()
df_features.head(10)

,Open,High,Low,Close,Volume,log_return,vol_5,vol_10,vol_21,vol_ewma,ATR,RSI,MACD,MACD_signal,log_volume,volume_change,target_vol
Date,,,,,,,,,,,,,,,,,
2015-02-03 00:00:00-05:00,2022.709961,2050.300049,2022.709961,2050.030029,4615900000,0.014336,0.014115,0.011876,0.011401,0.011663,30.628584,54.949410,-0.174383,-0.104848,22.252773,0.141132,0.010659
2015-02-04 00:00:00-05:00,2048.860107,2054.739990,2036.719971,2041.510010,4141920000,-0.004165,0.011980,0.011928,0.010659,0.011154,29.445007,55.609976,0.788912,0.075233,22.144425,-0.108347,0.010632
2015-02-05 00:00:00-05:00,2043.449951,2063.550049,2043.449951,2062.520020,3821990000,0.010239,0.012071,0.011342,0.010632,0.010990,28.885010,62.843383,2.954017,0.654408,22.064037,-0.080388,0.010417
2015-02-06 00:00:00-05:00,2062.280029,2072.399902,2049.969971,2055.469971,4232970000,-0.003424,0.009040,0.011249,0.010417,0.010524,28.177150,57.146245,4.116070,1.350026,22.166170,0.102133,0.009699
2015-02-09 00:00:00-05:00,2053.469971,2056.159912,2041.880005,2046.739990,3549540000,-0.004256,0.009016,0.011295,0.009699,0.010109,27.450718,54.691067,4.364544,1.955216,21.990084,-0.176086,0.009796
2015-02-10 00:00:00-05:00,2049.379883,2070.860107,2048.620117,2068.590088,3669850000,0.010619,0.007882,0.010778,0.009796,0.010085,27.298584,56.750971,6.041058,2.774861,22.023417,0.033333,0.009595
2015-02-11 00:00:00-05:00,2068.550049,2073.479980,2057.989990,2068.530029,3596860000,-0.000029,0.007295,0.009374,0.009595,0.009583,25.673575,51.124890,7.272450,3.676559,22.003327,-0.020090,0.009738
2015-02-12 00:00:00-05:00,2069.979980,2088.530029,2069.979980,2088.479980,3788350000,0.009598,0.007132,0.009382,0.009738,0.009442,26.201442,57.398263,9.574355,4.858404,22.055196,0.051869,0.009604
2015-02-13 00:00:00-05:00,2088.780029,2097.030029,2086.699951,2096.989990,3527450000,0.004066,0.006316,0.007425,0.009604,0.008988,25.750009,57.948180,11.872490,6.263396,21.983841,-0.071355,0.009252


In [29]:
df.shape, df_features.shape

((2242, 18), (2242, 17))